
# ASSIGNMENT 10  
# Comparing GAN and VAE Using Your Own Experimental Results

**Objective:** Compare a Generative Adversarial Network (GAN) and a Variational Autoencoder (VAE) using the generated images from the previous assignments.

### Required previous outputs
Put your images into these two folders before running the notebook:

- `GAN_Images/` → GAN-generated images from **Assignment 4**
- `VAE_Images/` → VAE-generated images from **Assignment 8**

The notebook will automatically select approximately **8 images from each model** and create the required comparison.


In [ ]:

# Install/import required libraries
import os
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageOps
from pathlib import Path

print("Libraries loaded successfully.")



## 1. Set the image folders

Keep this notebook and the two folders in the same directory.

Expected structure:

```text
Assignment_10_GAN_vs_VAE_Comparison.ipynb
GAN_Images/
    gan1.png
    gan2.png
    ...
VAE_Images/
    vae1.png
    vae2.png
    ...
```

Supported image formats: PNG, JPG, JPEG, BMP, WEBP.


In [ ]:

GAN_FOLDER = Path("GAN_Images")
VAE_FOLDER = Path("VAE_Images")

SUPPORTED_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".webp"}

def get_image_files(folder):
    if not folder.exists():
        return []
    return sorted([
        p for p in folder.iterdir()
        if p.is_file() and p.suffix.lower() in SUPPORTED_EXTENSIONS
    ])

gan_files = get_image_files(GAN_FOLDER)
vae_files = get_image_files(VAE_FOLDER)

print(f"GAN images found: {len(gan_files)}")
print(f"VAE images found: {len(vae_files)}")

if len(gan_files) < 8 or len(vae_files) < 8:
    print("\nNOTE: The assignment asks for approximately 8 images from each model.")
    print("Add more images to the corresponding folder if needed.")


## 2. Select 8 images from each model

In [ ]:

def select_images(files, n=8):
    if len(files) == 0:
        return []
    if len(files) <= n:
        return files

    # Evenly spaced selection so that different parts of the generated set are represented.
    indices = np.linspace(0, len(files) - 1, n, dtype=int)
    return [files[i] for i in indices]

gan_selected = select_images(gan_files, 8)
vae_selected = select_images(vae_files, 8)

print("Selected GAN images:")
for p in gan_selected:
    print(" -", p.name)

print("\nSelected VAE images:")
for p in vae_selected:
    print(" -", p.name)


## 3. Display the selected generated images

In [ ]:

def display_image_grid(files, title, rows=2, cols=4):
    if not files:
        print(f"No images found for {title}.")
        return

    fig, axes = plt.subplots(rows, cols, figsize=(14, 7))
    axes = np.array(axes).reshape(-1)

    for ax in axes:
        ax.axis("off")

    for i, file in enumerate(files[:rows * cols]):
        try:
            img = Image.open(file).convert("RGB")
            axes[i].imshow(img)
            axes[i].set_title(f"{i+1}. {file.name}", fontsize=9)
        except Exception as e:
            axes[i].text(0.5, 0.5, f"Error\n{file.name}", ha="center", va="center")
            print(f"Could not open {file}: {e}")

    fig.suptitle(title, fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()

display_image_grid(gan_selected, "GAN-Generated Images (Assignment 4)")
display_image_grid(vae_selected, "VAE-Generated Images (Assignment 8)")



## 4. Basic experimental measurements

The following measurements provide quantitative support for the comparison:

- **Sharpness:** estimated using the variance of the Laplacian. Higher values generally indicate stronger edges/detail.
- **Diversity:** estimated from average pairwise pixel difference after resizing images to a common size. Higher values indicate more variation within a set.

These measurements are only supporting evidence. **Realism, training stability, controllability, and mode-collapse risk should be judged from your actual previous experiments and visual outputs.**


In [ ]:

try:
    import cv2
    CV2_AVAILABLE = True
except ImportError:
    CV2_AVAILABLE = False
    print("OpenCV is not installed. Sharpness will use a simple gradient-based fallback.")

def load_gray(file, size=(128, 128)):
    img = Image.open(file).convert("L")
    img = ImageOps.fit(img, size)
    return np.asarray(img, dtype=np.float32)

def sharpness_score(file):
    arr = load_gray(file)

    if CV2_AVAILABLE:
        return float(cv2.Laplacian(arr, cv2.CV_32F).var())

    # Fallback: mean squared first-order gradient magnitude
    gx = np.diff(arr, axis=1)
    gy = np.diff(arr, axis=0)
    return float((gx[:-1, :]**2).mean() + (gy[:, :-1]**2).mean())

def diversity_score(files):
    if len(files) < 2:
        return 0.0

    arrays = [load_gray(f).flatten() / 255.0 for f in files]
    values = []

    for i in range(len(arrays)):
        for j in range(i + 1, len(arrays)):
            values.append(np.mean(np.abs(arrays[i] - arrays[j])))

    return float(np.mean(values))

def average_sharpness(files):
    if not files:
        return 0.0
    return float(np.mean([sharpness_score(f) for f in files]))

gan_sharpness = average_sharpness(gan_selected)
vae_sharpness = average_sharpness(vae_selected)

gan_diversity = diversity_score(gan_selected)
vae_diversity = diversity_score(vae_selected)

measurement_df = pd.DataFrame({
    "Metric": ["Average Sharpness", "Diversity Score"],
    "GAN": [gan_sharpness, gan_diversity],
    "VAE": [vae_sharpness, vae_diversity]
})

measurement_df



## 5. Normalize the quantitative measurements for easier comparison

The normalized scores below are relative scores from **0 to 10** for the two models in this experiment.

They do **not** claim that one model is universally better; they only compare the outputs placed in the two folders.


In [ ]:

def relative_scores(a, b):
    total = a + b
    if total == 0:
        return 5.0, 5.0
    return round(10 * a / total, 2), round(10 * b / total, 2)

gan_sharp_score, vae_sharp_score = relative_scores(gan_sharpness, vae_sharpness)
gan_div_score, vae_div_score = relative_scores(gan_diversity, vae_diversity)

quantitative_scores = pd.DataFrame({
    "Metric": ["Image Sharpness", "Image Diversity"],
    "GAN": [gan_sharp_score, gan_div_score],
    "VAE": [vae_sharp_score, vae_div_score]
})

quantitative_scores



## 6. Practical comparison based on the generated outputs

For the remaining criteria, enter scores from **1 (poor) to 10 (excellent)** based on what you observed during Assignments 4 and 8.

For **Mode Collapse Risk**, a higher number means **higher risk** (so lower is better).

The default values below are only placeholders. **Change them to match your actual results.**


In [ ]:

# ===== EDIT THESE VALUES TO MATCH YOUR OWN EXPERIMENTAL OBSERVATIONS =====

# 1 = poor, 10 = excellent
manual_scores = {
    "Realism": {"GAN": 7, "VAE": 6},
    "Training Stability": {"GAN": 5, "VAE": 8},
    "Ease of Controlling Output": {"GAN": 6, "VAE": 8},
    "Mode Collapse Risk": {"GAN": 7, "VAE": 3},  # Higher = MORE risk
}

# Merge quantitative results with manual observations
comparison_rows = [
    {
        "Criteria": "Image Sharpness",
        "GAN": gan_sharp_score,
        "VAE": vae_sharp_score,
        "Better": "GAN" if gan_sharp_score > vae_sharp_score else ("VAE" if vae_sharp_score > gan_sharp_score else "Tie")
    },
    {
        "Criteria": "Diversity",
        "GAN": gan_div_score,
        "VAE": vae_div_score,
        "Better": "GAN" if gan_div_score > vae_div_score else ("VAE" if vae_div_score > gan_div_score else "Tie")
    }
]

for metric, scores in manual_scores.items():
    if metric == "Mode Collapse Risk":
        better = "GAN" if scores["GAN"] < scores["VAE"] else ("VAE" if scores["VAE"] < scores["GAN"] else "Tie")
    else:
        better = "GAN" if scores["GAN"] > scores["VAE"] else ("VAE" if scores["VAE"] > scores["GAN"] else "Tie")

    comparison_rows.append({
        "Criteria": metric,
        "GAN": scores["GAN"],
        "VAE": scores["VAE"],
        "Better": better
    })

comparison_table = pd.DataFrame(comparison_rows)
comparison_table



## 7. Required Comparison Table

The table below is the main submission table. It combines the measured results with the practical observations from the two previous assignments.


In [ ]:

final_comparison = comparison_table.copy()

# Add interpretation column for the table
interpretations = {
    "Image Sharpness": "Higher score means sharper/more detailed images.",
    "Diversity": "Higher score means more variation between generated images.",
    "Realism": "Higher score means images look more realistic and closer to the training data.",
    "Training Stability": "Higher score means training was easier to keep stable.",
    "Ease of Controlling Output": "Higher score means output characteristics were easier to control.",
    "Mode Collapse Risk": "Higher score means greater risk of producing very similar outputs."
}

final_comparison["Interpretation"] = final_comparison["Criteria"].map(interpretations)
final_comparison



## 8. Overall Score

For the overall score, all criteria except Mode Collapse Risk are treated as **higher-is-better**.

For Mode Collapse Risk, the score is inverted so that **lower risk contributes more** to the overall result.


In [ ]:

score_df = final_comparison.copy()

gan_values = []
vae_values = []

for _, row in score_df.iterrows():
    if row["Criteria"] == "Mode Collapse Risk":
        gan_values.append(10 - float(row["GAN"]))
        vae_values.append(10 - float(row["VAE"]))
    else:
        gan_values.append(float(row["GAN"]))
        vae_values.append(float(row["VAE"]))

gan_overall = round(np.mean(gan_values), 2)
vae_overall = round(np.mean(vae_values), 2)

overall_df = pd.DataFrame({
    "Model": ["GAN", "VAE"],
    "Overall Score (out of 10)": [gan_overall, vae_overall]
})

overall_df



## 9. Final Verdict

The cells below generate a short assignment-ready verdict from the experimental comparison.

### Decision 1: Synthetic training data
This notebook uses a practical rule:
- Prefer the model with the stronger combination of **realism + sharpness + diversity**.

### Decision 2: Anomaly detection
A VAE is generally a natural choice when the task depends on learning a compact representation and identifying samples that do not fit the learned data distribution. Your own outputs and reconstruction behavior should also be considered.


In [ ]:

# Determine the preferred model for synthetic data
synthetic_scores = {
    "GAN": np.mean([
        float(final_comparison.loc[final_comparison["Criteria"] == "Image Sharpness", "GAN"].iloc[0]),
        float(final_comparison.loc[final_comparison["Criteria"] == "Diversity", "GAN"].iloc[0]),
        float(final_comparison.loc[final_comparison["Criteria"] == "Realism", "GAN"].iloc[0]),
    ]),
    "VAE": np.mean([
        float(final_comparison.loc[final_comparison["Criteria"] == "Image Sharpness", "VAE"].iloc[0]),
        float(final_comparison.loc[final_comparison["Criteria"] == "Diversity", "VAE"].iloc[0]),
        float(final_comparison.loc[final_comparison["Criteria"] == "Realism", "VAE"].iloc[0]),
    ])
}

synthetic_winner = max(synthetic_scores, key=synthetic_scores.get)

# For anomaly detection, start with VAE because its latent/reconstruction framework is commonly suitable.
anomaly_winner = "VAE"

print("FINAL VERDICT")
print("=" * 60)
print(f"1. Preferred model for generating synthetic training data: {synthetic_winner}")
print(f"   Reason: In this experiment it had the stronger combined result for sharpness, diversity and realism.")
print()
print(f"2. Preferred model for anomaly detection: {anomaly_winner}")
print("   Reason: VAE learns a latent representation and can use reconstruction behavior to help identify unusual samples.")
print()
print("3. Overall conclusion:")
print(
    f"   The GAN and VAE showed different strengths in the experiment. "
    f"The selected GAN/VAE images, quantitative measurements, and observation scores "
    f"should be used together rather than relying only on theory."
)



# Conclusion

Based on the generated images from Assignment 4 and Assignment 8, the GAN and VAE showed different strengths. The GAN generally focuses on producing visually sharp and realistic samples, while the VAE provides a more stable latent representation and is well suited to tasks where reconstruction and anomaly identification are important.

For **synthetic training data**, the preferred model should be the one that produced the better combination of realism, sharpness, and diversity in the actual experiment. For **anomaly detection**, the VAE is the preferred choice because its encoder-decoder structure provides a useful latent representation and reconstruction-based way to identify unusual samples.

**Important:** The scores in Section 6 are editable placeholders. Replace them with observations from your own Assignment 4 and Assignment 8 outputs before submitting.



## Submission Checklist

Before uploading:

- [ ] 8 GAN-generated images from Assignment 4
- [ ] 8 VAE-generated images from Assignment 8
- [ ] Comparison table completed
- [ ] Manual observation scores updated
- [ ] Final verdict checked
- [ ] All notebook cells run successfully
